In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os
import sys
import json
import random

import numpy as np 
import pandas as pd 

import librosa as lb
import librosa.feature as lf
import librosa.display as ld
import soundfile as sf
import kagglehub 


import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm

import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset,DataLoader

# Kaggle Set-up
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
SR = 22050
DURATION = 30

#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")


#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

import warnings
warnings.filterwarnings("ignore")

🚀 Using device: cuda
GPU is available.Setting RANDOM_SEED .... 
   GPU: Tesla T4
   Memory: 14.6 GB

✅ Environment setup complete!


In [2]:
class TestMusicDataset(Dataset):
    """
        This dataset takes file_paths as a list of all 15000
        paths of musics and load each music as waveform.
    """
    #,root_dir_path,file_names,sample_rate=SR
    def __init__(self,audios):
        self.audios = audios
    def __len__(self):
        return len(self.audios)
    def __getitem__(self,idx):
        mashup_id = self.audios[idx][0]
        waveform = self.audios[idx][1]
        return mashup_id,waveform

def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    waveform, _sr_ = torchaudio.load(path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if _sr_ != sr:
        resampler = torchaudio.transforms.Resample(_sr_, sr)
        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)

    # Trim or pad
    if waveform.shape[0] >= LENGTH:
        return waveform[:LENGTH]
    else:
        padding = LENGTH - waveform.shape[0]
        return torch.nn.functional.pad(waveform, (0, padding))


def extract_features_batch(waveforms, sr=22050):
    """
    waveforms: Tensor (batch_size, time) on CUDA
    returns: Tensor (batch_size, feature_dim)
    """
    waveforms = waveforms.squeeze(1)
    device = waveforms.device
    batch_size = waveforms.shape[0]
    # STFT (batched)
    stft = torch.stft(
        waveforms,
        n_fft=2048,
        hop_length=512,
        return_complex=True
    )  # (B, F, T)

    magnitude = stft.abs()
    power = magnitude ** 2

    # Frequency bins
    freqs = torch.linspace(0, sr/2, magnitude.shape[1], device=device)
    freqs = freqs.view(1, -1, 1)  # (1, F, 1)

    eps = 1e-8
    magnitude_sum = magnitude.sum(dim=1) + eps

    # Spectral Centroid
    centroid = (freqs * magnitude).sum(dim=1) / magnitude_sum
    centroid_mean = centroid.mean(dim=1)
    centroid_var = centroid.var(dim=1)

    # Spectral Bandwidth
    centroid_expanded = centroid.unsqueeze(1)
    bandwidth = torch.sqrt(
        ((freqs - centroid_expanded) ** 2 * magnitude).sum(dim=1)
        / magnitude_sum
    )
    bandwidth_mean = bandwidth.mean(dim=1)
    bandwidth_var = bandwidth.var(dim=1)

    #Spectral Rolloff (85%)
    cumulative = torch.cumsum(magnitude, dim=1)
    threshold = 0.85 * cumulative[:, -1:, :]
    rolloff = (cumulative >= threshold).float().argmax(dim=1)

    rolloff_mean = rolloff.float().mean(dim=1)
    rolloff_var = rolloff.float().var(dim=1)

    # RMS
    rms = torch.sqrt(torch.mean(waveforms ** 2, dim=1))

    # ZCR
    zcr = ((waveforms[:, 1:] * waveforms[:, :-1]) < 0).float().mean(dim=1)

    # MFCC (batched)
    mfcc_transform = T.MFCC(sample_rate=sr, n_mfcc=20).to(device)

    # MFCC expects (B, T)
    mfcc = mfcc_transform(waveforms)  # (B, 20, frames)

    mfcc_mean = mfcc.mean(dim=2)
    mfcc_var = mfcc.var(dim=2)

    # Concatenate All Features
    features = torch.cat([
        centroid_mean.unsqueeze(1),
        centroid_var.unsqueeze(1),
        bandwidth_mean.unsqueeze(1),
        bandwidth_var.unsqueeze(1),
        rolloff_mean.unsqueeze(1),
        rolloff_var.unsqueeze(1),
        rms.unsqueeze(1),
        zcr.unsqueeze(1),
        mfcc_mean,
        mfcc_var
    ], dim=1)

    return features         
print("✅")

✅


In [3]:
test_csv_path = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
sub_file_path = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv"
root_audio_path = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/"


test_df = pd.read_csv(test_csv_path)

audios = []
for idx,file_name in tqdm(zip(test_df['id'],test_df['filename']),total=test_df.shape[0],desc="Loading test mashup ...."):
    waveform = load_and_fix(root_audio_path+file_name)
    audios.append((idx,waveform))

Loading test mashup ....: 100%|██████████| 3020/3020 [01:39<00:00, 30.24it/s]


In [4]:
test_dataset = TestMusicDataset(audios)
test_data_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

all_features = []
all_ids = []
i = 0
for ids,waveforms in tqdm(test_data_loader,desc="Extracting features ....",total=len(test_data_loader)):
    waveforms = waveforms.to(device)
    features = extract_features_batch(waveforms)
    all_features.append(features.cpu())
    all_ids.extend(ids)
    
    
all_features = torch.cat(all_features, dim=0)
print(all_features.size())
print(len(all_ids))

Extracting features ....: 100%|██████████| 95/95 [00:10<00:00,  9.18it/s]

torch.Size([3020, 48])
3020


In [5]:
feature_mat = all_features.numpy()
ids_mat = np.array(all_ids).reshape(-1, 1)
print(feature_mat.shape,ids_mat.shape)

(3020, 48) (3020, 1)


In [6]:
def get_features_name():
    features_name = [
        "centroid_mean",
        "centroid_var",
        "bandwidth_mean",
        "bandwidth_var",
        "rolloff_mean",
        "rolloff_var",
        "rms",
        "zcr"
    ]
    for i in range(1,21):
        features_name.append(f"mfcc_mean{i}")
    for i in range(1,21):
        features_name.append(f"mfcc_var{i}")
        
    return features_name

id_df = pd.DataFrame(ids_mat,columns=['song_id'])
df = pd.DataFrame(feature_mat,columns=get_features_name())
final_df = pd.concat([id_df,df],axis=1)
print("✅", final_df.shape)

✅ (3020, 49)


In [7]:
final_df.isna().sum()

song_id           0
centroid_mean     0
centroid_var      0
bandwidth_mean    0
bandwidth_var     0
rolloff_mean      0
rolloff_var       0
rms               0
zcr               0
mfcc_mean1        0
mfcc_mean2        0
mfcc_mean3        0
mfcc_mean4        0
mfcc_mean5        0
mfcc_mean6        0
mfcc_mean7        0
mfcc_mean8        0
mfcc_mean9        0
mfcc_mean10       0
mfcc_mean11       0
mfcc_mean12       0
mfcc_mean13       0
mfcc_mean14       0
mfcc_mean15       0
mfcc_mean16       0
mfcc_mean17       0
mfcc_mean18       0
mfcc_mean19       0
mfcc_mean20       0
mfcc_var1         0
mfcc_var2         0
mfcc_var3         0
mfcc_var4         0
mfcc_var5         0
mfcc_var6         0
mfcc_var7         0
mfcc_var8         0
mfcc_var9         0
mfcc_var10        0
mfcc_var11        0
mfcc_var12        0
mfcc_var13        0
mfcc_var14        0
mfcc_var15        0
mfcc_var16        0
mfcc_var17        0
mfcc_var18        0
mfcc_var19        0
mfcc_var20        0
dtype: int64

In [9]:
final_df.to_csv(f'test-tabular-48-3020.csv',index=False)
if os.path.exists("/kaggle/working/test-tabular-48-3020.csv"):
    print("✅")

✅


In [10]:
# Storing to kaggle hub
handle = f'akashkumbhakar/test-tabular-48-3020-csv'
local_dataset= f'/kaggle/working/test-tabular-48-3020.csv'

# Create a new dataset
kagglehub.dataset_upload(handle, local_dataset)

Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/test-tabular-48-3020-csv ...
Starting upload for file /kaggle/working/test-tabular-48-3020.csv


Uploading: 100%|██████████| 1.47M/1.47M [00:00<00:00, 3.76MB/s]

Upload successful: /kaggle/working/test-tabular-48-3020.csv (1MB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/test-tabular-48-3020-csv
